In [ ]:
import os
import glob
import random
import math
import csv

from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

from torchvision import transforms
from torchvision.transforms import v2
from transformers import AutoModelForImageSegmentation

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfFolder, whoami

%matplotlib inline

In [ ]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
PLANTVILLAGE_PATH = '/kaggle/input/plantvillage-dataset/color'
# Background pool (one folder per class); download it with scripts/download_scraped.py
NEW_PLANTDOC_PATH = '/kaggle/input/background-pool/data'
SEGMENTED_DIR = '/kaggle/working/data'
BATCH_SIZE = 8
MODEL_NAME = 'briaai/RMBG-2.0'
IMAGE_SIZE = (1024, 1024)
MAX_FOREGROUND_IMGS = 5
TRACING_CSV = 'tracing.csv'

In [ ]:
user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("hf_token")
HfFolder.save_token(hf_token)

user = whoami()

In [ ]:
plant_village_map = {
    'Apple___healthy': 'Apple___healthy',
    'Apple___Cedar_apple_rust': 'Apple___rust',
    'Pepper,_bell___Bacterial_spot': 'Bell_pepper___bacterial_spot',
    'Pepper,_bell___healthy': 'Bell_pepper___healthy',
    'Grape___Black_rot': 'Grape___black_rot',
    'Grape___healthy': 'Grape___healthy',
    'Tomato___healthy': 'Tomato___healthy',
    'Tomato___Septoria_leaf_spot': 'Tomato___septoria_leaf_spot'
}

In [ ]:
class BGDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.classes = sorted(os.listdir(root_dir))
        self.index_to_class = {i: class_name for i, class_name in enumerate(self.classes)}
        self.image_paths = []
        self.labels = []
        
        for label, class_name in enumerate(self.classes):
            class_dir = os.path.join(root_dir, class_name)
            for img_path in glob.glob(os.path.join(class_dir, '*.png')) + \
                           glob.glob(os.path.join(class_dir, '*.jpg')) + \
                           glob.glob(os.path.join(class_dir, '*.jpeg')):
                self.image_paths.append(img_path)
                self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image =Image.open(img_path).convert("RGB")
        label = self.labels[idx]
        image = v2.ToTensor()(image)
        return image, label


In [ ]:
yolo_aug = v2.Compose([
    v2.RandomHorizontalFlip(p=0.1),
    v2.RandomVerticalFlip(p=0.1),
    v2.RandomPerspective(distortion_scale=0.6, p=1.0),
    v2.RandomAffine(degrees=(30, 70), translate=(0.1, 0.3), scale=(0.5, 0.75)),
    v2.RandomRotation(degrees=(0, 180)),
])

In [ ]:
class RemBG2(nn.Module):
    def __init__(self, bg_image_size, model_name, device):
        super(RemBG2, self).__init__()
        self.bg_image_size = bg_image_size
        self.model = AutoModelForImageSegmentation.from_pretrained(model_name, trust_remote_code=True).to(device)
        self.transform_image = transforms.Compose([
            transforms.Resize(bg_image_size),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        self.device = device
        self.new_image_size = (512, 512)

    def forward(self, img_paths, bg_img_path):
        background_pil = Image.open(bg_img_path).convert("RGBA")  # Ensure RGBA
        
        for img_path in img_paths:
            main_image_pil = Image.open(img_path).convert("RGB")  # Open each foreground image
            main_image_tensor = self.transform_image(main_image_pil).unsqueeze(0).to(self.device)

            with torch.no_grad():
                mask_logits = self.model(main_image_tensor)[-1]
                mask_probs = torch.sigmoid(mask_logits).squeeze(0).cpu()

            mask_resized = transforms.ToPILImage()(mask_probs)
            mask_resized = mask_resized.resize(main_image_pil.size, Image.Resampling.BILINEAR)
            main_image_pil.putalpha(mask_resized)  # Add transparency mask

            transformed_img = yolo_aug(main_image_pil)  # Apply augmentation

            background_pil = background_pil.resize(self.new_image_size)
            # Paste foreground onto background using transparency mask

            # Randomly scale foreground image
            scale_factor = random.uniform(1.5, 2.0)  # 30% to 80% of original size
            new_w = int(transformed_img.width * scale_factor)
            new_h = int(transformed_img.height * scale_factor)
            transformed_img = transformed_img.resize((new_w, new_h), Image.Resampling.LANCZOS)

            # Random position within background
            x_offset = random.randint(int(0 - new_w // 2), int(self.new_image_size[0] - new_w // 2))
            y_offset = random.randint(int(0 - new_h // 2), int(self.new_image_size[1] - new_h // 2))

            # Paste foreground onto background with transparency mask
            background_pil.paste(transformed_img, (x_offset, y_offset), transformed_img)
            
        return background_pil  # Final composited image

In [ ]:
bgDataset = BGDataset(NEW_PLANTDOC_PATH)
bg_data_loader = DataLoader(bgDataset, batch_size=BATCH_SIZE, shuffle=True)
rem_bg2 = RemBG2(IMAGE_SIZE, MODEL_NAME, DEVICE)

In [ ]:
def save_image(image, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    if path.lower().endswith('.jpg') or path.lower().endswith('.jpeg'):
        image = image.convert("RGB")  # ✅ Remove the alpha channel

    image.save(path)
    print(f"Saved final image to {path}")

In [ ]:
header = ['foreground', 'backgound', 'final', 'class']
with open(TRACING_CSV, 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(header)

In [ ]:
foreground_itr = int(round(math.sqrt(MAX_FOREGROUND_IMGS)))
print(foreground_itr)
for class_ in plant_village_map.keys():
    standard_name = plant_village_map[class_]
    source_path = os.path.join(PLANTVILLAGE_PATH, class_)
    bg_dir = os.path.join(NEW_PLANTDOC_PATH, standard_name)
    for i in range(foreground_itr):
        source_imgs = os.listdir(source_path)
        bg_imgs = os.listdir(bg_dir)
        for fg_img in source_imgs:
            fg_img_paths = []
            fg_img_path = os.path.join(source_path, fg_img)
            fg_img_paths.append(fg_img_path)
            bg_img_path = os.path.join(bg_dir, random.choice(bg_imgs))
            random_n = random.randint(0, MAX_FOREGROUND_IMGS-1) 
            if random_n > 0:
                random_fg_imgs = random.sample(source_imgs, random_n)
                for fg_img in random_fg_imgs:
                    fg_img_paths.append(os.path.join(source_path, fg_img))      
            try:
                save_img_name = f"{Path(fg_img).stem}_{i}.jpg"
                final_image = rem_bg2(fg_img_paths, bg_img_path)
                save_path = os.path.join(SEGMENTED_DIR, standard_name, save_img_name)
                save_image(final_image, save_path)
                row_data = [fg_img_paths, bg_img_path, save_path, class_]
                with open(TRACING_CSV, 'a', newline='') as file:
                    writer = csv.writer(file)
                    writer.writerow(row_data)
            
            except:
                print(f"Failed to change background of the {standard_name} image")